## Running this notebook

Install the project with the notebook extra before opening this file. Imports
use the installed package and do not change the working directory. Set
`OVERSCREENING_OUTPUT_DIR` to choose a directory for saved files.

The original scientific parameter values and sweep sizes are retained. Several
cells are deliberately expensive or demonstrate a failed solve; they are not
part of the bounded CI examples. Inspect convergence flags before interpreting
results. The optional fsolve initializer and native adjoint fallback have separate
controls; reference continuation remains a third, independent control. After a failed
physical Newton solve it computes a screened linear reference, then follows
the full residual from `q=1` to the physical endpoint `q=0`, retaining accepted
states and reducing the parameter step after failed stages. Source callbacks
remain physical and independent of `q`.
For bounded NumPy/mpmath demonstrations, run the scripts in `examples/`.

Ordinary BSK examples construct a fresh immutable `BSKProblem`; there are no
parameter setters in that interface. The initial problem defaults retain
sigma=10, mu=0, gamma=0.5, delta=50 and wall value 100. Every later parameter
block specifies its physical values and boundary data explicitly. Finite
segments use `domain` and both endpoint boundary rows. Unsupported boundary experiments have a separate opt-in research notebook.

Live solver diagnostics use `verbosity=2` in this notebook. Set it to `0`
for quiet solves, `1` for phase summaries, or `3` for trial details.


## This notebook provides a set of examples on how to use the numerical method for  problems on a sement.

In [ ]:
# Install the project first: python -m pip install -e ".[notebook]"
from pathlib import Path
import os
from matplotlib import pyplot as plt
import numpy as np
import mpmath as mp
from overscreening_breakdown import (
    BSKProblem, MpmathArithmetic, solve, load_solution,
    ContinuationOptions, ScreenedLinearReference,
)
from overscreening_breakdown.rational_mapping import RationalHalfLineMap
OUTPUT_DIR = Path(os.environ.get("OVERSCREENING_OUTPUT_DIR", "notebook-output")).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
%matplotlib inline

def display_values(function, coordinates):
    # Float conversion is confined to plot data.
    return np.array([float(function(value)) for value in coordinates])

def mapped_values(result, angles):
    a = result.arithmetic
    if result.domain[1] == a.inf:
        mapping = RationalHalfLineMap(a, L=result.L)
        coordinates = [mapping.from_angle(a.scalar(value)) for value in angles]
    else:
        left, right = result.domain
        coordinates = [(a.cos(a.scalar(value))*(right-left)+(right+left))/2 for value in angles]
    return display_values(result.u, coordinates)

def run_collocation(problem, N, *, L=1, tolerance="1e-10", method="newton",
                    max_iterations=50, globalization=True, continuation=None, initial=None,
                    use_mpmath=False, precision=100, initializer=None,
                    initializer_tolerance="0.1", initializer_max_evaluations=200,
                    newton_fallback=None, adjoint_max_steps=20, verbosity=2,
                    progress_callback=None, progress_interval=1.0):
    if initial is not None and hasattr(initial, "coefficients"):
        initial = (initial.coefficients if initial.N == N
                   else lambda x, a, previous=initial: previous.u(x))
    return solve(problem, N, solver="collocation", L=L, tolerance=tolerance,
                 method=method, max_iterations=max_iterations,
                 use_globalization=globalization, continuation=continuation,
                 initial_guess=initial,
                 use_mpmath=use_mpmath, precision=precision,
                 initializer=initializer, initializer_tolerance=initializer_tolerance,
                 initializer_max_evaluations=initializer_max_evaluations,
                 newton_fallback=newton_fallback, adjoint_max_steps=adjoint_max_steps,
                 verbosity=verbosity, progress_callback=progress_callback,
                 progress_interval=progress_interval)


In [ ]:
problem = BSKProblem(domain=(0, 50), boundary_conditions=((100, None, None, 0, None), (0, 0, None, None, None)))

In [ ]:
#set up the problem.
#If a parameter is not given, then a default value is used instead. 
#See the source file of "overscreening_breakdown" for more details for defauls values

sigma = 20.0
mu = 0.36
gamma = 1
delta = 10.0
u0 = 1.0
u0xxx = 0

#use some function as initial guess for the root in the Newton's method.
def some_func(x, a):
    return u0 * a.exp(-x)

#apply parameters
problem = BSKProblem(sigma=sigma, mu=mu, gamma=gamma, delta=delta, boundary_conditions=((u0, None, None, u0xxx, None), (0, 0, None, None, None)), domain=(0, 50), initial_guess=some_func)

In [ ]:
N = 100
tolerance = 1.0e-10
use_method = "newton"
initializer = None


In [ ]:
#pass the problem to the solver

# solver.set_problem(problem)
print("domain:", problem.domain) #check, that we are loading a version with the finite segment domain
print("boundary:",problem.boundary_conditions) #and with correct boundary conditions

In [ ]:
result = run_collocation(problem, N, tolerance=tolerance, method=use_method)
c = result.coefficients


In [ ]:
# Compare the optional NumPy fsolve initializer through the same public solve API.
initializer = "fsolve"
result_ao = run_collocation(problem, N, tolerance=tolerance, method=use_method,
                            initializer=initializer)
c_ao = result_ao.coefficients


In [ ]:
t = np.arange(0, np.pi, 0.005)
plt.title(r"finite-domain solution on mapped $[0,\pi]$")
plt.plot(t, mapped_values(result, t), '.')
plt.plot(t, mapped_values(result_ao, t))
plt.legend(["no fsolve initialization", "fsolve initialization"])
plt.show()
x = np.arange(0.0, 100.0, 0.1)
# Evaluate the finite solution only on its actual [0,50] domain. Retain the
# original plotting grid and explicitly mask the points outside that domain.
inside = x <= float(result.domain[1])
plt.title("solution on its physical [0,50] domain")
plt.plot(x[inside], display_values(result.u, x[inside]), '.')
plt.plot(x[inside], display_values(result_ao.u, x[inside]))
plt.legend(["no fsolve initialization", "fsolve initialization"])
plt.show()


Let's check the inifinite domain variant and compare it side by side

In [ ]:
problem_inf = BSKProblem()
sigma = 20.0
mu = 0.36
gamma = 1
delta = 10.0
u0 = 1.0
u0xxx = 0
def some_func(x, a):
    return u0 * a.exp(-x)
problem_inf = BSKProblem(sigma=sigma, mu=mu, gamma=gamma, delta=delta, boundary_conditions=((u0, None, None, u0xxx, None), (None,)*5), initial_guess=some_func)


In [ ]:
N = 100
tolerance = 1.0e-10
use_method = "newton"
domain = [0,np.inf]
initializer = "fsolve"
L = 6
print("domain:", problem_inf.domain)
print("boundary:", problem_inf.boundary_conditions)


In [ ]:
# The physical description is passed directly to the solve call below.


In [ ]:
result_inf = run_collocation(problem_inf, N, L=L, tolerance=tolerance,
                              method=use_method, initializer=initializer)
c_inf = result_inf.coefficients


In [ ]:
x_max = 50
x = np.arange(0.0, x_max, x_max/1000.0)
plt.title("solution on 0 <= x <= {:.0f}".format(x_max))
plt.plot(x, display_values(result.u, x), '.')
plt.plot(x, display_values(result_inf.u, x))
plt.legend(["finite segment", r"$\mathbb{R}^+$"])
plt.show()


Let's solve some other problems on a segment...

In [ ]:
# to be continued